# 03 — 共享逐股网络、掩码与真实数据接口

这个 notebook 检查两个关键性质：股票换槽后策略结果相应换槽；padding 和不可投资股票不参与 PPO 的动作概率。最后给出真实 NPZ 的构造模板。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/res/rl_experimental').exists())
sys.path.insert(0, str(repo_root))

from src.res.rl_experimental.data import PanelData, make_synthetic_panel  # noqa: E402
from src.res.rl_experimental.env import EpisodeRange, PortfolioEnv  # noqa: E402
from src.res.rl_experimental.policy import MaskedDiagonalGaussian, MaskedStockActorCriticPolicy  # noqa: E402
from src.res.rl_experimental.portfolio import PortfolioConstraints  # noqa: E402

panel = make_synthetic_panel(80, 30, 5, seed=23)
env = PortfolioEnv(panel, EpisodeRange(0, 50), PortfolioConstraints(10, 5, .25))
obs, _ = env.reset(seed=1)
policy = MaskedStockActorCriticPolicy(env.observation_space, env.action_space, lambda _: 3e-4, hidden_dim=16)
tensor_obs = {k: torch.as_tensor(v).unsqueeze(0) for k, v in obs.items()}

共享逐股参数意味着网络学习“具有某些特征的股票如何处理”，而不是把第 3 个输出永久绑定给某个代码。

In [ ]:
with torch.no_grad():
    mean, value = policy.network(tensor_obs)
perm = np.arange(env.constraints.max_slots)[::-1].copy()
permuted = {k: (v[:, perm] if k != 'account' else v) for k, v in tensor_obs.items()}
with torch.no_grad():
    perm_mean, perm_value = policy.network(permuted)
print('max action equivariance error:', float((mean[:, perm] - perm_mean).abs().max()))
print('critic value difference:', float((value - perm_value).abs().max()))

In [ ]:
# 改变一个被掩码槽位的动作，不应改变 PPO log probability。
mask = torch.tensor([[1.0, 0.0, 1.0]])
dist = MaskedDiagonalGaussian(torch.zeros(1, 3, 2), torch.zeros(1, 1, 2), mask)
a = torch.zeros(1, 3, 2)
b = a.clone()
b[:, 1] = 100.0
print('log probability A:', float(dist.log_prob(a)))
print('log probability B:', float(dist.log_prob(b)))

## 真实数据接入模板

下面的变量应由 learndl 的 point-in-time 数据流程产生。这里不连接数据库，目的是明确最终边界和形状。尤其不要用今天的指数成分股回填历史，也不要让 `forward_returns` 进入特征。

In [ ]:
def build_panel(stock_ids, decision_dates, return_end_dates, features, alpha, investable, forward_returns):
    panel = PanelData(
        stock_ids=np.asarray(stock_ids),
        decision_dates=np.asarray(decision_dates),
        return_end_dates=np.asarray(return_end_dates),
        features=np.asarray(features, dtype=np.float32),
        alpha=np.asarray(alpha, dtype=np.float32),
        investable=np.asarray(investable, dtype=bool),
        forward_returns=np.asarray(forward_returns, dtype=np.float32),
    )
    print(f'T={panel.n_steps}, N={panel.n_stocks}, F={panel.n_features}')
    return panel

# 用合成数组演示接口；替换参数即可接入真实数据。
example = build_panel(
    panel.stock_ids, panel.decision_dates, panel.return_end_dates,
    panel.features, panel.alpha, panel.investable, panel.forward_returns,
)

In [ ]:
# 准备好真实数据后保存，并使用 CLI 训练：
# example.save_npz('panel.npz')
# python -m src.res.rl_experimental train panel.npz --output results/rl_experimental/real-v1

checks = {
    'unique stock IDs': len(np.unique(example.stock_ids)) == example.n_stocks,
    'end date not before decision': bool(np.all(example.return_end_dates >= example.decision_dates)),
    'finite eligible returns': bool(np.isfinite(example.forward_returns[example.investable]).all()),
}
checks